# From Combine Cuts to Game Performance: Kinematic Cut Efficiency (KCE)

**NFL Big Data Bowl 2027 · Open Track**

This public notebook accompanies the Kaggle Writeup. It inspects the official Combine tracking input directly, verifies why the shipped acceleration magnitude cannot measure deceleration, and independently derives signed kinematics from position. The reported model metrics are the saved five-fold out-of-fold results in the accompanying source artifact; the optional final cell reruns the complete pipeline. No raw competition data is repackaged in this notebook.

In [ ]:
import base64, json, os, sys, zipfile
from pathlib import Path
import pandas as pd

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = '...'
bundle = WORK / 'source_bundle.zip'
bundle.write_bytes(base64.b64decode(BUNDLE_B64))
with zipfile.ZipFile(bundle) as zf:
    zf.extractall(WORK)

# Discover official competition inputs without assuming Kaggle's nested folder layout.
file_map = {}
for directory, _subdirs, files in os.walk('/kaggle/input'):
    for name in files:
        if name in {'combine_tracking.csv', 'player_play.csv', 'game_tracking_2023.csv',
                    'combine_results.csv', 'player_career_successes.csv', 'players.csv', 'games.csv'}:
            file_map.setdefault(name, Path(directory) / name)
required = ['combine_tracking.csv', 'player_play.csv', 'game_tracking_2023.csv']
missing = [name for name in required if name not in file_map]
if missing:
    raise FileNotFoundError(f'Competition data missing: {missing}')
DATA = WORK / 'data/kaggle_nfl/nfl_data'
DATA.mkdir(parents=True, exist_ok=True)
for name, source_path in file_map.items():
    target = DATA / name
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source_path)

sys.path.insert(0, str(WORK / 'src'))
print('Official CSVs found:', sorted(file_map))
print('Competition source:', file_map['combine_tracking.csv'].parent)
print('Analysis modules:', len(list((WORK / 'src').glob('*.py'))))


## 1. Tracking-data check

Combine files are the actual competition inputs. The primary check differentiates position with a Savitzky–Golay filter and labels cuts from heading changes. This makes the distinction between alignment side in the drill name and physical break direction explicit.

In [ ]:
import numpy as np
import pandas as pd
from kce_features import savgol_kinematics, detect_cuts

# Read the actual Combine tracking file used by the report. The shipped `a` field is
# an unsigned magnitude; signed acceleration is recomputed from positions.
usecols = ['event_id', 'nfl_id', 'entity_type', 'time', 'drill_name', 'attempt', 'x', 'y', 'a']
ct = pd.read_csv(DATA / 'combine_tracking.csv', usecols=usecols, parse_dates=['time'])
ct = ct[ct.entity_type == 'PLAYER'].sort_values(['event_id', 'nfl_id', 'time'])
print(f'PLAYER rows: {len(ct):,} | athletes: {ct.nfl_id.nunique():,} | '
      f'shipped acceleration negative fraction: {(ct.a < 0).mean():.4%}')

# Independently re-derive signed longitudinal acceleration for a bounded, deterministic
# sample of real sessions; this is a validation cell, not a synthetic data path.
negative_fraction, session_count, cut_count = [], 0, 0
for (_event, _nfl_id), group in ct.groupby(['event_id', 'nfl_id'], sort=False):
    if len(group) < 11:
        continue
    x = group.x.to_numpy(dtype=float); y = group.y.to_numpy(dtype=float)
    kin = savgol_kinematics(x, y, dt=0.1)
    kin['x'] = x; kin['y'] = y
    negative_fraction.extend((kin['a_long'] < 0).tolist())
    session_count += 1
    cut_count += len(detect_cuts(kin, str(group.drill_name.iloc[0])))
    if session_count >= 100:
        break
print(f'Sampled sessions: {session_count} | signed derivative negative fraction: '
      f'{np.mean(negative_fraction):.2%} | detected cuts: {cut_count}')


## 2. Cross-validated results and leakage check

Scores below are grouped out-of-fold results. The primary official target is separation at pass-forward. The higher `sep_gain` score is reported with the no-cushion ablation because cushion is algebraically part of `sep_gain`; it must not be read as a clean headline without that qualification.

In [ ]:
import json
from pathlib import Path

metrics = json.loads((WORK / 'model_metrics.json').read_text())

def get_score(target, grouping, model='lightgbm'):
    return metrics[target]['groupings'][grouping]['models'][model]

rows = []
for target, grouping in [
    ('sep_at_pass_forward', 'by_player'),
    ('sep_at_pass_forward', 'by_game_play'),
    ('sep_gain', 'by_game_play'),
    ('sep_gain_no_cushion', 'by_game_play'),
    ('career_snaps_log', 'by_player'),
    ('yards_after_catch', 'by_game_play'),
]:
    result = get_score(target, grouping)
    rows.append({'target': target, 'grouping': grouping, 'OOF R²': result['r2'],
                 'MAE': result['mae'], 'Spearman ρ': result['spearman']})
summary = pd.DataFrame(rows)
print('Stored metrics: five-fold out-of-fold; grouped by athlete or game-play as labeled.')
display(summary.round(4))
print('Main target is separation at pass-forward. sep_gain includes cushion algebraically; '
      'the no-cushion ablation is shown separately.')


In [ ]:
from IPython.display import Image, display
from pathlib import Path
figure_names = ["fig1_cross_product_inside_outside.png", "fig5_feature_importance.png", "fig6_oof_pred_vs_actual.png"]
for name in figure_names:
    matches = list(Path("/kaggle/input").rglob(name))
    if matches:
        print(name)
        display(Image(filename=str(matches[0])))
    else:
        print("Figure not found:", name)


## 3. Reproduction

The notebook bundles the analysis source code and aggregate metric artifact; the source bundle contains no raw league or player tracking files. Kaggle supplies the competition data as the only raw-data input. Set `RUN_FULL_CV=1` to rebuild tables, rerun grouped cross-validation, mixed-effects checks, and figures. The default fast path verifies the raw tracking derivation and displays the saved OOF report so judges can inspect the evidence without waiting for the full rerun.

In [ ]:
# Full reproducing analysis (optional; computes all five-fold models and mixed effects).
# Keep RUN_FULL_CV unset for a fast notebook read-through; set it to `1` to rerun.
if os.environ.get('RUN_FULL_CV') == '1':
    from build_tables import main as build_tables
    import run_analysis
    build_tables()
    run_analysis.main()
else:
    print('Full model rerun is opt-in. Set RUN_FULL_CV=1 to rebuild all real-data caches, '
          'five-fold models, mixed-effects checks and figures.')
    print('Source is bundled in /kaggle/working/src; raw data remains the competition input.')
